In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/datasets/qnightnguyen/car-topview/12525572-hd_1920_1080_30fps.mp4
/kaggle/input/datasets/qnightnguyen/car-topview/3968162649-preview.mp4
/kaggle/input/datasets/qnightnguyen/car-topview/3692969835-preview.mp4
/kaggle/input/datasets/qnightnguyen/car-topview/3942949981-preview.mp4
/kaggle/input/datasets/qnightnguyen/car-topview/3759509695-preview.mp4
/kaggle/input/datasets/qnightnguyen/car-topview/1039561247-preview.mp4
/kaggle/input/datasets/qnightnguyen/car-topview/3942940165-preview.mp4
/kaggle/input/datasets/qnightnguyen/car-topview/Car-topview-segmentation/Car-topview-segmentation/README.dataset.txt
/kaggle/input/datasets/qnightnguyen/car-topview/Car-topview-segmentation/Car-topview-segmentation/README.roboflow.txt
/kaggle/input/datasets/qnightnguyen/car-topview/Car-topview-segmentation/Car-topview-segmentation/data.yaml
/kaggle/input/datasets/qnightnguyen/car-topview/Car-topview-segmentation/Car-topview-segmentation/valid/labels/9999989_00000_d_0000001_jpg.rf.f7ff2b8fb

In [2]:
! pip install torch torchsummary

In [ ]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.double_conv(x)

class UNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=1):
        super(UNet, self).__init__()
        self.inc = DoubleConv(n_channels, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))
        # self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(512, 1024)) # Đáy (Bottleneck)


        self.up1 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv_up1 = DoubleConv(512, 256)
        
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv_up2 = DoubleConv(256, 128)
        
        self.up3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv_up3 = DoubleConv(128, 64)
        
        # Up 4: 128 -> 64. Sau đó concat với inc (64) -> tổng 128
        # self.up4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        # self.conv_up4 = DoubleConv(128, 64)
        
        self.outc = nn.Conv2d(64, n_classes, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        # x5 = self.down4(x4) # Bottleneck
        
        x = self.up1(x4)
        x = torch.cat([x, x3], dim=1)
        x = self.conv_up1(x)
        
        x = self.up2(x)
        x = torch.cat([x, x2], dim=1)
        x = self.conv_up2(x)
        
        x = self.up3(x)
        x = torch.cat([x, x1], dim=1)
        x = self.conv_up3(x)
        
        # x = self.up4(x)
        # x = torch.cat([x, x1], dim=1)
        # x = self.conv_up4(x)
        
        return self.outc(x)

In [ ]:
model = UNet()
from torchsummary import summary
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
summary(model, input_size=(3, 256, 256))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 256, 256]           1,792
       BatchNorm2d-2         [-1, 64, 256, 256]             128
              ReLU-3         [-1, 64, 256, 256]               0
            Conv2d-4         [-1, 64, 256, 256]          36,928
       BatchNorm2d-5         [-1, 64, 256, 256]             128
              ReLU-6         [-1, 64, 256, 256]               0
        DoubleConv-7         [-1, 64, 256, 256]               0
         MaxPool2d-8         [-1, 64, 128, 128]               0
            Conv2d-9        [-1, 128, 128, 128]          73,856
      BatchNorm2d-10        [-1, 128, 128, 128]             256
             ReLU-11        [-1, 128, 128, 128]               0
           Conv2d-12        [-1, 128, 128, 128]         147,584
      BatchNorm2d-13        [-1, 128, 128, 128]             256
             ReLU-14        [-1, 128, 1

In [ ]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset
import torch.nn.functional as F

class CarDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        self.transform = transform

    def __len__(self): 
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        
        mask_name = os.path.splitext(img_name)[0] + ".png"
        mask_path = os.path.join(self.mask_dir, mask_name)
        
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        if not isinstance(mask, torch.Tensor):
            mask = torch.from_numpy(mask).float()
        
        if mask.ndim == 2:
            mask = mask.unsqueeze(0)
            
        mask = mask / 255.0 if mask.max() > 1.0 else mask
        
        return image, mask

class DiceLoss(torch.nn.Module):
    def __init__(self, smooth=1e-6):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        # Quan trọng: Đưa đầu ra model về dạng xác suất [0, 1]
        pred = torch.sigmoid(pred)
        
        # Flatten
        iflat = pred.view(-1)
        tflat = target.view(-1)
        
        intersection = (iflat * tflat).sum()
        dice = (2. * intersection + self.smooth) / (iflat.sum() + tflat.sum() + self.smooth)
        return 1 - dice

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_transform = A.Compose([
    A.Resize(256, 256),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Rotate(limit=15, p=0.5),
    
    A.Affine(
        translate_percent={"x": (-0.05, 0.05), "y": (-0.05, 0.05)}, 
        scale=(0.95, 1.05), 
        rotate=(-15, 15), 
        p=0.5
    ),
    
    A.RGBShift(r_shift_limit=15, g_shift_limit=15, b_shift_limit=15, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
], is_check_shapes=False)

In [10]:
# Dùng DataLoader của bạn
train_ds = CarDataset(
    img_dir="/kaggle/input/datasets/qnightnguyen/car-topview/Car-topview-segmentation/Car-topview-segmentation/train/images", 
    mask_dir="/kaggle/input/datasets/qnightnguyen/car-topview/Car-topview-segmentation/Car-topview-segmentation/train/mask",
    transform=train_transform
)
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader


class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        
        # Flatten tensor
        probs = probs.view(-1)
        targets = targets.view(-1)
        
        intersection = (probs * targets).sum()
        dice = (2. * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 1 - dice


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet(n_channels=3, n_classes=1).to(device)

criterion_dice = DiceLoss() 
criterion_bce = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(model.parameters(), lr=1e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

def compute_loss(outputs, masks):
    bce = criterion_bce(outputs, masks)
    dice = criterion_dice(outputs, masks)
    return bce + dice # Tổng hợp cả 2 để bắt object nhỏ tốt hơn

for epoch in range(100):
    model.train()
    epoch_loss = 0
    
    for imgs, masks in train_loader:
        imgs = imgs.to(device, dtype=torch.float32)
        masks = masks.to(device, dtype=torch.float32)

        outputs = model(imgs)
        
        loss = compute_loss(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    
    scheduler.step(avg_loss)
    
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch [{epoch+1}/100], Loss: {avg_loss:.4f}, LR: {current_lr}")

# Lưu model
torch.save(model.state_dict(), "/kaggle/working/unet_car_seg.pth")

Epoch [1/100], Loss: 1.0437, LR: 0.0001
Epoch [2/100], Loss: 0.8204, LR: 0.0001
Epoch [3/100], Loss: 0.7385, LR: 0.0001
Epoch [4/100], Loss: 0.6645, LR: 0.0001
Epoch [5/100], Loss: 0.6004, LR: 0.0001
Epoch [6/100], Loss: 0.5436, LR: 0.0001
Epoch [7/100], Loss: 0.4885, LR: 0.0001
Epoch [8/100], Loss: 0.4452, LR: 0.0001
Epoch [9/100], Loss: 0.4015, LR: 0.0001
Epoch [10/100], Loss: 0.3714, LR: 0.0001
Epoch [11/100], Loss: 0.3346, LR: 0.0001
Epoch [12/100], Loss: 0.3113, LR: 0.0001
Epoch [13/100], Loss: 0.2870, LR: 0.0001
Epoch [14/100], Loss: 0.2652, LR: 0.0001
Epoch [15/100], Loss: 0.2487, LR: 0.0001
Epoch [16/100], Loss: 0.2358, LR: 0.0001
Epoch [17/100], Loss: 0.2191, LR: 0.0001
Epoch [18/100], Loss: 0.2119, LR: 0.0001
Epoch [19/100], Loss: 0.1962, LR: 0.0001
Epoch [20/100], Loss: 0.1895, LR: 0.0001
Epoch [21/100], Loss: 0.1818, LR: 0.0001
Epoch [22/100], Loss: 0.1730, LR: 0.0001
Epoch [23/100], Loss: 0.1676, LR: 0.0001
Epoch [24/100], Loss: 0.1622, LR: 0.0001
Epoch [25/100], Loss: 0.1

In [ ]:
import cv2
import numpy as np
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_model(model_path):
    model = UNet().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    return model

def predict_video(video_path, model, output_path='output_video.mp4'):
    cap = cv2.VideoCapture(video_path)
    
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = cap.get(cv2.CAP_PROP_FPS)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    print(f"🚀 Đang xử lý video từ: {video_path}...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        img_input = cv2.resize(frame, (256, 256))
        img_input = cv2.cvtColor(img_input, cv2.COLOR_BGR2RGB)
        img_input = img_input / 255.0
        
        img_input = torch.from_numpy(img_input).permute(2, 0, 1).float().unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(img_input)
            prob = torch.sigmoid(output) 
            mask_pred = prob.squeeze().cpu().numpy()
            
            mask_final = (mask_pred > 0.5).astype(np.uint8) * 255

        mask_resized = cv2.resize(mask_final, (width, height))
        
        mask_rgb = np.zeros_like(frame)
        mask_rgb[:, :, 1] = mask_resized 


        combined_frame = cv2.addWeighted(frame, 0.7, mask_rgb, 0.3, 0)
        
        out.write(combined_frame)

    cap.release()
    out.release()
    print(f"✅ Xử lý xong! Video lưu tại: {output_path}")

model_path = "weights.pth"
video_input_path = "/kaggle/input/datasets/qnightnguyen/car-topview/3968162649-preview.mp4"

model = load_model(model_path)

predict_video(video_input_path, model)

🚀 Đang xử lý video từ: /kaggle/input/datasets/qnightnguyen/car-topview/3968162649-preview.mp4...
✅ Xử lý xong! Video lưu tại: output_video.mp4
